<a href="https://colab.research.google.com/github/madanjha/Machine-Learning/blob/main/SVD12thOct2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**SVD (Singular Value Decomposition)**
It is a way to break a big matrix into 3 smaller parts that are easier to understand and work with

`A = U x E x V^T`
* A - Complete matrix
* U - Left Singular Vector (People's Prefernece -> Who likes what)
* E - Sigma - Singular value (Strength of pattern) -> How strong each pattern is
* V^T - Right Singular Vector (Movie Feature) -> What kind of movie it is

**import libraries**

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse.linalg import svds

**We will create the user item matrix**

In [ ]:
data = {
    'Singham': [5,4,np.nan,1],
    'Harry Potter': [1,np.nan,5,4],
    'Iron Man': [np.nan,1,5,4],
    'Titanic':[4,5,np.nan,2]
}

In [ ]:
data

{'Singham': [5, 4, nan, 1],
 'Harry Potter': [1, nan, 5, 4],
 'Iron Man': [nan, 1, 5, 4],
 'Titanic': [4, 5, nan, 2]}

In [ ]:
users = ['Harendra','Aryan','sudipta','Gudimani']
df = pd.DataFrame(data,users)

In [ ]:
df

,Singham,Harry Potter,Iron Man,Titanic
Harendra,5.0,1.0,NaN,4.0
Aryan,4.0,NaN,1.0,5.0
sudipta,NaN,5.0,5.0,NaN
Gudimani,1.0,4.0,4.0,2.0


we will use `.values` becuse we want array on which we can perform 0

In [ ]:
ratingDf = df.fillna(0).values

In [ ]:
ratingDf

array([[5., 1., 0., 4.],
       [4., 0., 1., 5.],
       [0., 5., 5., 0.],
       [1., 4., 4., 2.]])

**Now we will apply SVD to reduce the dimension**

Here the sigma we will give us (1D array)

In [ ]:
k = 2 # Latent Features (hiddne feature)
U , Sigma, Vt = svds(ratingDf,k=k)

In [ ]:
U

array([[-0.48826253, -0.49479723],
       [-0.48278898, -0.49992914],
       [ 0.64501596, -0.47277054],
       [ 0.33536384, -0.53078675]])

In [ ]:
Sigma

array([ 7.95423773, 10.26785258])

In [ ]:
Vt

array([[-0.50754137,  0.51271697,  0.5134051 , -0.46469159],
       [-0.48739397, -0.48518392, -0.48568372, -0.53958781]])

Since the sigma is 1D of singular value, We need to turn it into a diaognal matrix for matrix multiplication

In [ ]:
sigma_diag = np.diag(Sigma)

**Reconstruct the matrix to predict Ratings**

The new matrix after reconstruction will contain the predicted Ratings for the empty space that we had

In [ ]:
predictRatingMatrix = np.dot(np.dot(U,sigma_diag),Vt)

In [ ]:
predictRatingMatrix

array([[ 4.44737451,  0.4737116 ,  0.47357832,  4.54612745],
       [ 4.45095978,  0.52160034,  0.52152336,  4.5543286 ],
       [-0.23802184,  4.98579777,  4.99175452,  0.23519027],
       [ 1.30241776,  4.011977  ,  4.01653657,  1.70118082]])

**Let's Make table**

In [ ]:
ratingDf

array([[5., 1., 0., 4.],
       [4., 0., 1., 5.],
       [0., 5., 5., 0.],
       [1., 4., 4., 2.]])

In [ ]:
predictRatingMatrixDF = pd.DataFrame(predictRatingMatrix,index = users, columns = df.columns)

In [ ]:
df

,Singham,Harry Potter,Iron Man,Titanic
Harendra,5.0,1.0,NaN,4.0
Aryan,4.0,NaN,1.0,5.0
sudipta,NaN,5.0,5.0,NaN
Gudimani,1.0,4.0,4.0,2.0


In [ ]:
predictRatingMatrixDF.round(2)

,Singham,Harry Potter,Iron Man,Titanic
Harendra,4.45,0.47,0.47,4.55
Aryan,4.45,0.52,0.52,4.55
sudipta,-0.24,4.99,4.99,0.24
Gudimani,1.30,4.01,4.02,1.70


**Now, let's make recommendation**

In [ ]:
def recommendItems(userID, originalDF, PredictionDF, numRecommendation = 5):
  # Recomend the items for a give user
  userRowNumber = originalDF.index.get_loc(userID) # here we will get the user's row number
  sortedUserPrediction = PredictionDF.iloc[userRowNumber].sort_values(ascending= True) # it will give us user's predicted rating
  userOriginalRating = originalDF.loc[userID] # it Will give us the user's original rating to filter out items that they have already seen
  recommendation = userOriginalRating[userOriginalRating.isnull()] # We are recommending items that the user has not yet rated

  # Now we will merge the predictions and sort it out
  recommendation = recommendation.to_frame('OriginalRatings').merge(
      sortedUserPrediction.to_frame('PredictedRating'),
      left_index = True,
      right_index = True
  )

  return recommendation.head(numRecommendation)

**Let's get some recommendations 'harendra'**

In [ ]:
harendraRecommendation = recommendItems('Harendra',df,predictRatingMatrixDF)

In [ ]:
harendraRecommendation

,OriginalRatings,PredictedRating
Iron Man,NaN,0.473578


In [ ]:
aryanRecommendations = recommendItems('Aryan',df,predictRatingMatrixDF)

In [ ]:
aryanRecommendations

,OriginalRatings,PredictedRating
Harry Potter,NaN,0.5216


###**Advantage**
* It works for any matrix not just square
* It also help us in Dimensionality Reduction
* it is also usefull for image comprression

###**disadvantage**
* When we have a lot of data -> it will use also some of the resources

In [ ]:
U , Sigma, Vt

(array([[-0.48826253, -0.49479723],
        [-0.48278898, -0.49992914],
        [ 0.64501596, -0.47277054],
        [ 0.33536384, -0.53078675]]),
 array([ 7.95423773, 10.26785258]),
 array([[-0.50754137,  0.51271697,  0.5134051 , -0.46469159],
        [-0.48739397, -0.48518392, -0.48568372, -0.53958781]]))

In [ ]:
U @ sigma_diag @ Vt # ---> Return you the same matrix (A)

array([[ 4.44737451,  0.4737116 ,  0.47357832,  4.54612745],
       [ 4.45095978,  0.52160034,  0.52152336,  4.5543286 ],
       [-0.23802184,  4.98579777,  4.99175452,  0.23519027],
       [ 1.30241776,  4.011977  ,  4.01653657,  1.70118082]])

Documentation -> https://surpriselib.com/ To implement Any kind of Recommendation System